In [6]:
import pandas as pd
import urllib.request
import zipfile
import os

# Download dataset zip
url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
zip_path = "bank.zip"

urllib.request.urlretrieve(url, zip_path)

# Extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data")

print("✅ Downloaded and extracted")

✅ Downloaded and extracted


In [10]:
import zipfile

with zipfile.ZipFile("data/bank-additional.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

print("✅ Extracted inner zip")

✅ Extracted inner zip


In [11]:
df = pd.read_csv("data/bank-additional/bank-additional-full.csv", sep=";")

print("Shape:", df.shape)
df.head()

Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [12]:
#Upload data to S3
import boto3

bucket = "mlops-sagemaker-v1"

df.to_csv("bank.csv", index=False)

s3 = boto3.client("s3")

s3.upload_file("bank.csv", bucket, "raw/bank.csv")

print(f"✅ Uploaded to s3://{bucket}/raw/bank.csv")

✅ Uploaded to s3://mlops-sagemaker-v1/raw/bank.csv


In [4]:
!pip uninstall -y sagemaker
!pip install "sagemaker[local]"==2.224.0

Found existing installation: sagemaker 3.13.0
Uninstalling sagemaker-3.13.0:
  Successfully uninstalled sagemaker-3.13.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.1 MB/s  0:00:00


  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.31.1
    Uninstalling protobuf-6.31.1:
      Successfully uninstalled protobuf-6.31.1


   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/8 [ppft]

  Attempting uninstall: cloudpickle
   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/8 [ppft]

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 7/8 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 7/8 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 7/8 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 7/8 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.73.1 requires protobuf<7.0.0,>=6.30.0, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.42.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
sagemaker-studio-analytics-extension 0.3.0 requires sparkmagic==0.22.0, but you have sparkmagic 0.21.0 which is incompatible.
sparkmagic 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.3.3 which is incompatible.


In [5]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
import sagemaker

bucket = "mlops-sagemaker-v1"

session = sagemaker.Session()
role = sagemaker.get_execution_role()

processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.t3.medium",
    instance_count=1
)

processor.run(
    code="src/preprocessing.py",
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/raw/bank.csv",
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
            destination=f"s3://{bucket}/processed/train"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
            destination=f"s3://{bucket}/processed/test"
        )
    ],
    arguments=[
        "--input-data", "/opt/ml/processing/input/bank.csv",
        "--output-train", "/opt/ml/processing/output/train",
        "--output-test", "/opt/ml/processing/output/test"
    ]
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2026-06-05-06-49-25-667


ClientError: An error occurred (ValidationException) when calling the CreateProcessingJob operation: No S3 objects found under S3 URL "s3://mlops-sagemaker-v1/raw/bank.csv" given in input data source. Please ensure that the bucket exists in the selected region (us-east-1), that objects exist under that S3 prefix, and that the role "arn:aws:iam::928747699677:role/service-role/AmazonSageMaker-ExecutionRole-20260605T113958" has "s3:ListBucket" permissions on bucket "mlops-sagemaker-v1". Error message from S3: The bucket is in this region: ap-south-1. Please use this region to retry the request